In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score,mean_absolute_percentage_error

In [3]:
df = pd.read_csv(r"C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\final_dataset.csv", parse_dates=["Date"])

df = df.dropna()
df.head()

,Date,KCL,Phosphate,DAP,Gas_EUR,Brent
0,1967-01-01,25.5,12.0,68.5,0.456647,1.33
1,1967-02-01,25.5,12.0,68.5,0.456647,1.33
2,1967-03-01,25.5,12.0,68.5,0.456647,1.33
3,1967-04-01,25.5,12.0,68.5,0.456647,1.33
4,1967-05-01,25.5,12.0,68.5,0.456647,1.33


In [4]:
# feature engineering
#variables calendrier
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter

In [5]:
#lag features
lags = [1, 2, 3, 6, 12]

for lag in lags:
    df[f"KCL_lag_{lag}"] = df["KCL"].shift(lag)

In [8]:
#moyennes mobiles
df["KCL_roll_mean_3"] = (
    df["KCL"].shift(1)
    .rolling(window=3)
    .mean()
)

df["KCL_roll_mean_12"] = (
    df["KCL"].shift(1)
    .rolling(window=12)
    .mean()
)
df = df.dropna()


In [9]:
df.isna().sum()

Date                0
KCL                 0
Phosphate           0
DAP                 0
Gas_EUR             0
Brent               0
Year                0
Month               0
Quarter             0
KCL_lag_1           0
KCL_lag_2           0
KCL_lag_3           0
KCL_lag_6           0
KCL_lag_12          0
KCL_roll_mean_3     0
KCL_roll_mean_12    0
dtype: int64

In [10]:
features = [

    # vars economiques
    "Phosphate",
    "DAP",
    "Gas_EUR",
    "Brent",

    # vars temporelles
    "Year",
    "Month",
    "Quarter",

    # historique kcl
    "KCL_lag_1",
    "KCL_lag_2",
    "KCL_lag_3",
    "KCL_lag_6",
    "KCL_lag_12",

    # tendnaces
    "KCL_roll_mean_3",
    "KCL_roll_mean_12"
]

In [11]:
X= df[features]
y = df["KCL"]

In [12]:
train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

In [13]:
model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

In [14]:
forecast = model.predict(X_test)

In [15]:
mae = mean_absolute_error(y_test, forecast)
mse = mean_squared_error(y_test, forecast)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_test, forecast)
r2 = r2_score(y_test, forecast)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"MAPE : {mape*100:.2f}%")
print(f"R²   : {r2:.4f}")

MAE  : 31.951
RMSE : 60.901
MAPE : 7.31%
R²   : 0.9112


In [16]:
print(X_train.columns)

Index(['Phosphate', 'DAP', 'Gas_EUR', 'Brent', 'Year', 'Month', 'Quarter',
       'KCL_lag_1', 'KCL_lag_2', 'KCL_lag_3', 'KCL_lag_6', 'KCL_lag_12',
       'KCL_roll_mean_3', 'KCL_roll_mean_12'],
      dtype='object')


In [17]:
print(df.iloc[train_size][["Date","KCL"]])

Date    2014-07-01 00:00:00
KCL                   279.5
Name: 570, dtype: object


In [18]:
print(df.iloc[train_size-1][["Date","KCL"]])

Date    2014-06-01 00:00:00
KCL                   301.5
Name: 569, dtype: object


In [19]:
df.iloc[train_size][
[
"KCL",
"KCL_lag_1",
"KCL_roll_mean_3",
"KCL_roll_mean_12"
]
]

KCL                      279.5
KCL_lag_1                301.5
KCL_roll_mean_3     300.458333
KCL_roll_mean_12       354.375
Name: 570, dtype: object

In [42]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=10)

model.fit(
    X_train,
    y_train
)

forecast = model.predict(X_test)


mae = mean_absolute_error(y_test, forecast)
mse = mean_squared_error(y_test, forecast)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_test, forecast)
r2 = r2_score(y_test, forecast)


print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"MAPE : {mape*100:.2f}%")
print(f"R²   : {r2:.4f}")

MAE  : 32.273
RMSE : 61.464
MAPE : 7.37%
R²   : 0.9083


In [20]:
from sklearn.linear_model import Ridge

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

ridge_results = []

for alpha in alphas:

    model = Ridge(alpha=alpha)

    model.fit(X_train, y_train)

    forecast = model.predict(X_test)

    mae = mean_absolute_error(y_test, forecast)
    mse = mean_squared_error(y_test, forecast)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_test, forecast)
    r2 = r2_score(y_test, forecast)

    ridge_results.append({
        "alpha": alpha,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    })

ridge_results = pd.DataFrame(ridge_results)

ridge_results = ridge_results.sort_values(
    by="RMSE"
)

print(ridge_results)

      alpha        MAE          MSE       RMSE      MAPE        R2
6  1000.000  28.171567  2625.104573  51.235774  0.067100  0.937164
5   100.000  30.177099  3206.780867  56.628446  0.069974  0.923241
4    10.000  31.674561  3634.892526  60.290070  0.072578  0.912993
3     1.000  31.922395  3701.173019  60.837267  0.073010  0.911407
2     0.100  31.948075  3708.150804  60.894588  0.073054  0.911240
1     0.010  31.950652  3708.852246  60.900347  0.073059  0.911223
0     0.001  31.950910  3708.922427  60.900923  0.073059  0.911221
